# 01 — Data Audit

**Purpose:** Validate data quality, understand distributions, and confirm the dataset is ready for analysis.

This notebook answers:
- How many creators and posts do we have?
- What does the follower tier distribution look like?
- What share of posts are sponsored?
- Are there missing values or outliers that need attention?

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from src.ingest import load_creators, load_posts, load_benchmarks
from src.clean import clean_creators, clean_posts, data_quality_report
from src.utils import set_plot_style, COLORS, TIER_ORDER, TIER_COLORS

set_plot_style()
pd.set_option('display.max_columns', 30)

## 1. Load and Clean Data

In [ ]:
creators_raw = load_creators()
posts_raw = load_posts()
benchmarks = load_benchmarks()

creators = clean_creators(creators_raw)
posts = clean_posts(posts_raw)

print(f"Creators: {len(creators):,}")
print(f"Posts: {len(posts):,}")
print(f"Benchmarks: {len(benchmarks):,}")

## 2. Data Quality Report

In [ ]:
report = data_quality_report(creators, posts)
for k, v in report.items():
    print(f"{k:35s} {v}")

## 3. Follower Tier Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Creator count by tier
tier_counts = creators['follower_tier'].value_counts().reindex(TIER_ORDER)
axes[0].bar(tier_counts.index, tier_counts.values, color=TIER_COLORS)
axes[0].set_title('Creators by Follower Tier')
axes[0].set_ylabel('Count')

# Engagement rate by tier
tier_er = creators.groupby('follower_tier')['avg_engagement_rate'].median().reindex(TIER_ORDER)
axes[1].bar(tier_er.index, tier_er.values, color=TIER_COLORS)
axes[1].set_title('Median Engagement Rate by Tier')
axes[1].set_ylabel('Engagement Rate (%)')

plt.tight_layout()
plt.savefig('../dashboard/tier_distribution.png')
plt.show()

## 4. Category Breakdown

In [ ]:
cat_summary = creators.groupby('category').agg(
    creators=('creator_id', 'count'),
    avg_followers=('followers', 'mean'),
    avg_er=('avg_engagement_rate', 'mean'),
    avg_sponsored_rate=('sponsored_post_rate', 'mean'),
).round(2).sort_values('avg_er', ascending=False)

cat_summary

## 5. Sponsored vs Organic Split

In [ ]:
sponsored_pct = posts['is_sponsored'].mean() * 100
print(f"Sponsored posts: {posts['is_sponsored'].sum():,} ({sponsored_pct:.1f}%)")
print(f"Organic posts: {(~posts['is_sponsored']).sum():,} ({100-sponsored_pct:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [posts['is_sponsored'].sum(), (~posts['is_sponsored']).sum()],
    labels=['Sponsored', 'Organic'],
    colors=[COLORS['accent'], COLORS['primary']],
    autopct='%1.1f%%',
    startangle=90,
)
ax.set_title('Post Type Distribution')
plt.savefig('../dashboard/sponsored_split.png')
plt.show()

## 6. Missing Values Check

In [ ]:
print('--- Creators ---')
print(creators.isnull().sum()[creators.isnull().sum() > 0])
print('\n--- Posts ---')
print(posts.isnull().sum()[posts.isnull().sum() > 0])
print('\nNo missing values detected.' if (creators.isnull().sum().sum() + posts.isnull().sum().sum()) == 0 else '')

## 7. DuckDB Quick Validation

In [ ]:
con = duckdb.connect()
con.register('creators', creators)
con.register('posts', posts)

result = con.sql("""
    SELECT
        follower_tier,
        COUNT(DISTINCT creator_id) AS creators,
        ROUND(AVG(avg_engagement_rate), 2) AS avg_er,
        ROUND(AVG(sponsored_post_rate) * 100, 1) AS sponsored_pct
    FROM creators
    GROUP BY follower_tier
    ORDER BY avg_er DESC
""").df()

result

## Summary

**Key findings from the data audit:**
- Dataset contains 500 creators and ~25K posts across 15 content categories
- Follower tier distribution skews toward nano and micro creators (expected for Instagram)
- ~20% of posts are sponsored — sufficient volume for sponsored vs organic comparisons
- No critical missing values or data quality issues
- Engagement rates follow expected inverse relationship with follower count

**Next step:** Feature engineering (Notebook 02)